# DecodeLabs Project 2 — Exploratory Data Analysis (EDA)

**Data Analytics Industrial Training | Project 2**

This notebook explores the dataset to uncover patterns, trends, distributions, relationships and outliers using descriptive statistics and visual analysis.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
sns.set_theme(style="whitegrid")

file_path = "../data/Project_2_EDA_Report_Final.xlsx"
df = pd.read_excel(file_path, sheet_name="Raw Data")
df.head()

## 1. Dataset Overview

Inspect the size, columns, data types and sample records.

In [ ]:
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])
print("\nColumns:", df.columns.tolist())
print("\nData types:")
print(df.dtypes)
df.head(10)

## 2. Data Quality Check

Check duplicate records, missing values and the date range.

In [ ]:
quality_summary = pd.DataFrame({
    "Metric": ["Total Records","Total Columns","Duplicate Rows","Missing Cells",
               "Unique Customers","Start Date","End Date"],
    "Value": [
        len(df), len(df.columns), df.duplicated().sum(),
        int(df.isna().sum().sum()), df["CustomerID"].nunique(),
        pd.to_datetime(df["Date"]).min(), pd.to_datetime(df["Date"]).max()
    ]
})
quality_summary

In [ ]:
missing_values = df.isna().sum().sort_values(ascending=False)
missing_values[missing_values > 0]

## 3. Descriptive Statistics

Calculate count, mean, median, standard deviation, minimum and maximum for the main numerical fields.

In [ ]:
numeric_cols = ["Quantity", "UnitPrice", "ItemsInCart", "TotalPrice"]

descriptive_stats = pd.DataFrame({
    "Count": df[numeric_cols].count(),
    "Mean": df[numeric_cols].mean(),
    "Median": df[numeric_cols].median(),
    "Std Dev": df[numeric_cols].std(),
    "Minimum": df[numeric_cols].min(),
    "Maximum": df[numeric_cols].max()
})
descriptive_stats

## 4. Product Analysis

In [ ]:
product_analysis = (
    df.groupby("Product")
      .agg(Order_Count=("OrderID","count"),
           Total_Quantity=("Quantity","sum"),
           Total_Revenue=("TotalPrice","sum"),
           Average_Order_Value=("TotalPrice","mean"))
      .sort_values("Total_Revenue", ascending=False)
)
product_analysis

In [ ]:
plt.figure(figsize=(10,6))
sns.barplot(data=product_analysis.reset_index(), x="Total_Revenue", y="Product")
plt.title("Revenue by Product")
plt.xlabel("Total Revenue")
plt.ylabel("Product")
plt.tight_layout()
plt.show()

## 5. Monthly Trend Analysis

In [ ]:
df["Date"] = pd.to_datetime(df["Date"])

monthly_trend = (
    df.groupby(df["Date"].dt.to_period("M"))
      .agg(Order_Count=("OrderID","count"), Total_Revenue=("TotalPrice","sum"))
      .reset_index()
)
monthly_trend["Date"] = monthly_trend["Date"].dt.to_timestamp()
monthly_trend

In [ ]:
plt.figure(figsize=(12,6))
sns.lineplot(data=monthly_trend, x="Date", y="Total_Revenue", marker="o")
plt.title("Monthly Revenue Trend")
plt.xlabel("Month")
plt.ylabel("Total Revenue")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 6. Category Analysis

Explore order status, payment method and referral source.

In [ ]:
status_analysis = df["OrderStatus"].value_counts().rename_axis("OrderStatus").reset_index(name="Count")
status_analysis["Percentage"] = status_analysis["Count"] / len(df) * 100
status_analysis

In [ ]:
plt.figure(figsize=(9,5))
sns.barplot(data=status_analysis, x="OrderStatus", y="Count")
plt.title("Order Status Distribution")
plt.xlabel("Order Status")
plt.ylabel("Number of Orders")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

In [ ]:
payment_analysis = df["PaymentMethod"].value_counts().rename_axis("PaymentMethod").reset_index(name="Count")
payment_analysis

In [ ]:
referral_analysis = (
    df.groupby("ReferralSource")
      .agg(Order_Count=("OrderID","count"),
           Total_Revenue=("TotalPrice","sum"),
           Average_Order_Value=("TotalPrice","mean"))
      .sort_values("Total_Revenue", ascending=False)
)
referral_analysis

In [ ]:
plt.figure(figsize=(10,6))
sns.barplot(data=referral_analysis.reset_index(), x="Total_Revenue", y="ReferralSource")
plt.title("Revenue by Referral Source")
plt.xlabel("Total Revenue")
plt.ylabel("Referral Source")
plt.tight_layout()
plt.show()

## 7. Distribution Analysis

In [ ]:
fig, axes = plt.subplots(1,3,figsize=(16,5))
sns.histplot(df["TotalPrice"], kde=True, ax=axes[0])
axes[0].set_title("TotalPrice Distribution")
sns.histplot(df["Quantity"], kde=False, ax=axes[1])
axes[1].set_title("Quantity Distribution")
sns.histplot(df["ItemsInCart"], kde=False, ax=axes[2])
axes[2].set_title("Items in Cart Distribution")
plt.tight_layout()
plt.show()

## 8. Outlier Detection — IQR Method

IQR = Q3 − Q1; upper bound = Q3 + 1.5 × IQR.

In [ ]:
Q1 = df["TotalPrice"].quantile(0.25)
Q3 = df["TotalPrice"].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = df[(df["TotalPrice"] < lower_bound) | (df["TotalPrice"] > upper_bound)].copy()

print("Q1:", Q1)
print("Q3:", Q3)
print("IQR:", IQR)
print("Lower Bound:", lower_bound)
print("Upper Bound:", upper_bound)
print("Number of Outliers:", len(outliers))

outliers[["OrderID","Product","Quantity","UnitPrice","TotalPrice"]].sort_values(
    "TotalPrice", ascending=False
)

In [ ]:
plt.figure(figsize=(10,5))
sns.boxplot(x=df["TotalPrice"])
plt.title("TotalPrice Outlier Detection")
plt.xlabel("TotalPrice")
plt.tight_layout()
plt.show()

## 9. Correlation Analysis

In [ ]:
corr_cols = ["Quantity","UnitPrice","ItemsInCart","TotalPrice"]
correlation_matrix = df[corr_cols].corr()
correlation_matrix

In [ ]:
plt.figure(figsize=(8,6))
sns.heatmap(correlation_matrix, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1)
plt.title("Correlation Matrix")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8,5))
sns.scatterplot(data=df, x="Quantity", y="TotalPrice")
plt.title("Quantity vs TotalPrice")
plt.xlabel("Quantity")
plt.ylabel("TotalPrice")
plt.tight_layout()
plt.show()

## 10. Key Observations

Use the calculated outputs above to write evidence-based observations. Consider: top product by revenue, highest/lowest revenue month, most common order status and payment method, top referral source, number of IQR outliers, strongest correlations, and notable distributions.

**Important:** If a year contains only a partial period, do not interpret its total as a full-year comparison without stating the limitation.

In [ ]:
top_product = product_analysis["Total_Revenue"].idxmax()
top_month = monthly_trend.loc[monthly_trend["Total_Revenue"].idxmax()]
low_month = monthly_trend.loc[monthly_trend["Total_Revenue"].idxmin()]

print(f"Top product by revenue: {top_product} ({product_analysis.loc[top_product,'Total_Revenue']:,.2f})")
print(f"Highest revenue month: {top_month['Date'].strftime('%Y-%m')} ({top_month['Total_Revenue']:,.2f})")
print(f"Lowest revenue month: {low_month['Date'].strftime('%Y-%m')} ({low_month['Total_Revenue']:,.2f})")
print(f"Most common order status: {status_analysis.iloc[0]['OrderStatus']} ({status_analysis.iloc[0]['Percentage']:.2f}% of orders)")
print(f"Most common payment method: {payment_analysis.iloc[0]['PaymentMethod']}")
print(f"Top referral source by revenue: {referral_analysis.index[0]}")
print(f"TotalPrice IQR outliers: {len(outliers)}")

## 11. Conclusion

This EDA examined descriptive statistics, product performance, monthly trends, category distributions, distributions, IQR outliers, correlations and visualizations. The resulting analysis turns the order dataset into interpretable findings about sales performance, trends, order behavior and unusual observations.